# 04 · Predict with S1 and score
Runs each exported S1 model on the calibration and test splits through the same sharded runner as the baselines, then scores and appends to the ledger. Nemotron is scored with the `no-nemotron` models only (prereg).

In [ ]:
# Shared session setup (run in every notebook)
import os, subprocess, pathlib
from google.colab import drive, userdata
drive.mount('/content/drive')
DRIVE = pathlib.Path('/content/drive/MyDrive/s1pii'); DRIVE.mkdir(parents=True, exist_ok=True)
os.environ['S1PII_DATA'] = '/content/s1pii_data'        # local disk; synced to/from Drive
os.environ['S1PII_RESULTS'] = str(DRIVE / 'results')     # ledger + prediction cache on Drive
REPO = pathlib.Path('/content/s1-pii')

def git(*args):
    # The token (Colab secret GH_TOKEN: fine-grained, read-only on anitksahu/s1-pii) is passed
    # through GIT_ASKPASS, never on the command line, and errors are printed redacted.
    try:
        tok = userdata.get('GH_TOKEN')
    except Exception:
        tok = None
    if not tok:   # public repo: no token needed
        r = subprocess.run(['git', *args], capture_output=True, text=True)
        if r.returncode: raise RuntimeError('git ' + args[0] + ' failed: ' + r.stderr[-800:])
        return r.stdout
    askpass = pathlib.Path('/tmp/askpass.sh'); askpass.write_text('#!/bin/sh\necho "$S1PII_GIT_TOKEN"\n'); askpass.chmod(0o700)
    env = {**os.environ, 'GIT_ASKPASS': str(askpass), 'S1PII_GIT_TOKEN': tok, 'GIT_TERMINAL_PROMPT': '0'}
    r = subprocess.run(['git', *args], env=env, capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError('git ' + args[0] + ' failed: ' + (r.stderr or '').replace(tok, '***')[-800:])
    return r.stdout

if not REPO.exists():
    git('clone', 'https://x-access-token@github.com/anitksahu/s1-pii.git', str(REPO))
else:
    git('-C', str(REPO), 'pull', '--ff-only')
os.chdir(REPO)
subprocess.run(['rsync', '-a', f'{DRIVE}/data/', os.environ['S1PII_DATA'] + '/'], check=False)
print(subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)


In [ ]:
!pip install -q -e '.[data,model]'

In [ ]:
import subprocess, json, os, pathlib
from s1pii import bench
MODELS = pathlib.Path('/content/drive/MyDrive/s1pii/models')
OUT = os.environ['S1PII_RESULTS'] + '/predictions'
DATASETS = ['tab_direct', 'spy_medical', 'spy_legal', 'pii_trace', 'nemotron']
def run_s1(model_dir, system, docs, extra=()):
    cmd = ['python', '-m', 's1pii.model.run_s1', '--model', str(model_dir), '--docs', str(docs), '--out', OUT, '--system', system, *extra]
    log = pathlib.Path(os.environ['S1PII_RESULTS']) / 'logs' / f"{system}-{pathlib.Path(docs).stem}.log"
    log.parent.mkdir(parents=True, exist_ok=True)
    last = ''
    with open(log, 'a') as fh:
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            fh.write(line); fh.flush(); last = line if line.startswith('{') else last
            if '] shard' in line: print(line, end='')
    if p.wait(): raise RuntimeError(f'{system} failed; see {log}')
    return json.loads(last)['predictions']
S1PRED = {}
for mdir in sorted(MODELS.glob('*/final')):
    variant_seed = mdir.parent.name
    for d in DATASETS:
        if d == 'nemotron' and not variant_seed.startswith('no-nemotron'):
            continue
        if d != 'nemotron' and variant_seed.startswith('no-nemotron'):
            continue
        p = bench.split_paths(d)
        S1PRED[(variant_seed, d)] = {part: run_s1(mdir, f's1_{variant_seed}', p[part]) for part in ('calib', 'test')}
        print(variant_seed, d, 'done')

In [ ]:
import pandas as pd
rows = []
for (vs, d), p in S1PRED.items():
    r = bench.score(f's1_{vs}', d, pathlib.Path(p['calib']), pathlib.Path(p['test']))['result']
    rows.append({'system': f's1_{vs}', 'dataset': d, 'pAUC': r['pauc'], 'pAUC_ci': r['pauc_ci'], 'reach_over': r['reachable_max_over']})
pd.DataFrame(rows).sort_values(['dataset', 'pAUC'])